In [2]:
%pip install -q langchain langchain_community langchain-gigachat langchain-huggingface chromadb langchain_chroma sentence_transformers pyTelegramBotAPI

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [3]:
from google.colab import files
import os
if not os.path.exists('digital_ruble.txt'):
    files.upload()

Saving digital_ruble.txt to digital_ruble.txt


In [4]:
doc = open('digital_ruble.txt', encoding='utf-8').read()
docs = [d.strip() for d in doc.split('\\\\') if d.strip()]
print(f"Всего фрагментов: {len(docs)}")
print(f"Макс длина фрагмента: {max(map(len, docs))} символов")
assert max(map(len, docs)) <= 1500, "Есть фрагмент длиннее 1500 символов"
print(docs[0])

Всего фрагментов: 26
Макс длина фрагмента: 588 символов
Цифровой рубль - это третья форма российского рубля наряду с наличными и безналичными деньгами. Его выпускает Банк России, и цифровой рубль является обязательством Центрального банка, а не коммерческого банка. Один цифровой рубль всегда равен одному наличному и одному безналичному рублю, обмен между формами происходит по курсу 1 к 1 без какой-либо конвертации.


In [6]:
import warnings
warnings.filterwarnings('ignore')
from google.colab import userdata
from langchain_chroma import Chroma

try:
    from langchain_gigachat.chat_models import GigaChat
except ImportError:
    from langchain_community.chat_models.gigachat import GigaChat
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings

GPT = GigaChat(
    credentials=userdata.get('gigachat_creds'),
    scope="GIGACHAT_API_PERS",
    model="GigaChat-2",
    streaming=False,
    verify_ssl_certs=False,
    temperature=0.3,
)

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    encode_kwargs={"normalize_embeddings": True},
)

print(GPT.invoke("Привет! Ответь одним словом: работаешь?").content)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Да


Модель e5 обучена с префиксами `passage:` для документов и `query:` для вопросов.

In [7]:
db = Chroma('digital_ruble', embeddings, persist_directory='./db')
if db._collection.count() == 0:
    db.add_texts(['passage: ' + d for d in docs])
print("Фрагментов в базе:", db._collection.count())

retriever = db.as_retriever(search_kwargs={"k": 4})

def search(q):
    return [x.page_content.removeprefix('passage: ') for x in retriever.invoke('query: ' + q)]

for frag in search("Начисляются ли проценты на цифровые рубли?"):
    print('-', frag[:150], '...')

Фрагментов в базе: 26
- На остаток цифровых рублей в кошельке не начисляются проценты. Банки не могут открывать вклады и выдавать кредиты в цифровых рублях. Цифровой рубль за ...
- Цифровой рубль - это третья форма российского рубля наряду с наличными и безналичными деньгами. Его выпускает Банк России, и цифровой рубль является о ...
- Цифровой рубль не является криптовалютой. У криптовалют нет единого эмитента, государственных гарантий и защиты прав потребителей, а их курс сильно ко ...
- Цифровой рубль сочетает свойства наличных и безналичных денег. Как безналичные деньги, он позволяет совершать удалённые платежи и переводы через мобил ...


In [8]:
SYSTEM = (
    "Ты - консультант по цифровому рублю. Отвечай на вопрос пользователя, опираясь "
    "только на факты из справочных фрагментов ниже. Делай выводы и объясняй простым языком, "
    "кратко (до 5-6 предложений), не цитируй фрагменты целиком. Если в фрагментах нет ответа, "
    "честно скажи, что в базе знаний нет такой информации. Не выдумывай цифры и даты."
)

def answer(q):
    context = '\n\n'.join(f'Фрагмент {i+1}: {t}' for i, t in enumerate(search(q)))
    prompt = f"{SYSTEM}\n\nСправочные фрагменты:\n{context}\n\nВопрос: {q}\nОтвет:"
    return GPT.invoke(prompt).content

test_questions = [
    "Чем цифровой рубль отличается от биткоина?",
    "Что будет с моими цифровыми рублями, если у банка отзовут лицензию?",
    "Обязан ли маленький магазин с выручкой 3 млн в год принимать цифровой рубль?",
    "Выгоднее ли хранить сбережения в цифровых рублях?",
    "Какой курс доллара сегодня?",
]
for q in test_questions:
    print('ВОПРОС:', q)
    print('ОТВЕТ:', answer(q), '\n')

ВОПРОС: Чем цифровой рубль отличается от биткоина?
ОТВЕТ: Цифровой рубль отличается от биткоина тем, что выпускается Банком России, имеет государственную гарантию и курс, равный обычному рублю. Биткоин не имеет единого эмитента и его курс сильно колеблется. Цифровой рубль отслеживается государством, а биткоин анонимен и децентрализован. 

ВОПРОС: Что будет с моими цифровыми рублями, если у банка отзовут лицензию?
ОТВЕТ: Если у банка отзовут лицензию, ваши цифровые рубли останутся в вашем цифровом кошельке на платформе Центрального банка. Вы сможете воспользоваться ими через другой коммерческий банк, поскольку сами цифровые рубли не хранятся непосредственно в банке. 

ВОПРОС: Обязан ли маленький магазин с выручкой 3 млн в год принимать цифровой рубль?
ОТВЕТ: Нет, маленький магазин с выручкой менее 5 млн рублей в год не обязан принимать цифровой рубль. Согласно правилам, освобождение от обязанности распространяется на компании с совсем небольшой выручкой. 

ВОПРОС: Выгоднее ли хранить сб

In [12]:
import telebot

bot = telebot.TeleBot(userdata.get('tg_token'))

@bot.message_handler(commands=['start', 'help'])
def start(message):
    bot.send_message(message.chat.id,
        'Привет! Меня зовут Циферка, я — ai-консультант по цифровому рублю. Спроси меня, например:\n'
        '• Чем цифровой рубль отличается от безналичных денег?\n'
        '• Сколько стоят операции для граждан и бизнеса?\n'
        '• Когда цифровой рубль начнут принимать магазины?')

@bot.message_handler(content_types=['text'])
def handle_message(message):
    bot.send_chat_action(message.chat.id, 'typing')
    try:
        ans = answer(message.text)
    except Exception as e:
        ans = 'Не получилось получить ответ от модели, попробуйте ещё раз.'
        print('Ошибка:', e)
    for i in range(0, len(ans), 4000):   # лимит Telegram - 4096 символов
        bot.send_message(message.chat.id, ans[i:i+4000])

print("Бот готов к работе")
bot.infinity_polling(skip_pending=True)

Бот готов к работе
